# From Analog Ritual to Digital Reality
### *Modern Strain Gage Technology* — Companion Notebook

Companion notebook to Chapter 10 — From Analog Ritual to Digital Reality.

The transition from analog to digital data acquisition transformed strain measurement more thoroughly than any other single development since the gage itself. An analog recorder's fidelity was limited by the chart speed and the analyst's ability to read the trace. A digital system's fidelity is determined by two numbers that must be set *before the test begins*: **sampling rate** and **bit depth**. Understanding both — and the failure modes that arise when they are chosen incorrectly — is a prerequisite for any modern strain measurement program.

The **sampling theorem**, established by Nyquist in 1928 and Shannon in 1949, states that a signal can be perfectly reconstructed from digital samples only if the sample rate exceeds twice the highest frequency component present in the signal. Below that rate, energy from high-frequency components folds back into the spectrum and appears as a lower-frequency signal — *aliasing*. The alias is indistinguishable from a real signal at that frequency; the only defense is an analog low-pass filter applied *before* the ADC, removing the offending frequencies before they reach the converter. This is not an optional refinement — it is mandatory in any measurement where signal content above f_s/2 cannot be ruled out in advance.

**Bit depth** determines dynamic range: how large a signal the system can represent relative to its smallest resolvable step. An 8-bit ADC at ±5 V resolves 39 mV per count — corresponding to thousands of microstrain. A 24-bit ADC at the same range resolves 0.6 μV, or approximately 0.24 με at 5 V excitation and GF = 2.0. Between these extremes, 16-bit and 18-bit converters define the practical range of modern strain measurement.

**This notebook demonstrates:**
1. **Aliasing** — a 1,000 Hz sinusoidal strain signal sampled adequately at 2,500 Hz (Nyquist = 1,250 Hz) versus under-sampled at 1,200 Hz (Nyquist = 600 Hz), where it folds down to a false 200 Hz alias. The signal and sample rates are configurable so you can explore other cases.
2. **Quantization error and ADC resolution** — dynamic range, LSB voltage, and strain resolution from 8 to 24 bits.

In [ ]:
# Colab setup — install dependencies (harmless no-op locally if already present)
# !pip install numpy matplotlib

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# Figures are written here — these paths match what the book \includegraphics.
OUT_DIR = Path("../src/media/ch10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Aliasing Demonstration: Signal Above and Below Nyquist

If a signal contains frequency components above the Nyquist frequency (f_s/2), those components *alias* — they fold back into the spectrum and appear at a lower, fictitious frequency. The alias is indistinguishable from a real signal at that frequency. No amount of post-processing can recover the correct signal once aliasing has occurred in the ADC.

The two-panel figure compares adequate and inadequate sampling of a 1,000 Hz sinusoidal strain signal:

- **Left panel:** Sampling at 2,500 Hz (Nyquist = 1,250 Hz > 1,000 Hz). The sample points closely track the true signal — reconstruction is faithful.
- **Right panel:** Sampling at 1,200 Hz (Nyquist = 600 Hz < 1,000 Hz). The same 1,000 Hz signal aliases to |1,000 − 1,200| = **200 Hz** — an entirely false low-frequency signal that cannot be distinguished from real structural response.

Change `f_signal`, `f_sample_ok`, and `f_sample_bad` to explore other scenarios. An analog anti-aliasing filter set below f_sample/2 is the only protection — it must be applied before the ADC, not in software.

In [ ]:
# --- Aliasing demonstration: 1,000 Hz signal, adequate vs. under-sampling ---
F_SIGNAL     = 1000.0  # true signal frequency (Hz)
F_SAMPLE_OK  = 2500.0  # adequate sampling rate (Hz) — above 2×F_SIGNAL
F_SAMPLE_BAD = 1200.0  # inadequate sampling rate (Hz) — below 2×F_SIGNAL
WINDOW_S     = 0.020   # display window (s) — 20 ms
N_CONT       = 4000    # points used to draw the "continuous" reference waves


def alias_frequency(f_signal, f_sample):
    """Return (f_alias, sign) for a pure tone sampled at f_sample.

    The tone at ``f_signal`` folds into the baseband [0, f_sample/2]. Each fold
    through the Nyquist frequency negates the reconstructed amplitude, so the
    returned ``sign`` (+1 or -1) is the amplitude flip applied to the apparent
    low-frequency tone. Below Nyquist the tone is unchanged (sign = +1).
    """
    omega_norm = (f_signal / f_sample) % 1.0  # normalized digital frequency, [0, 1)
    if omega_norm > 0.5:
        return (1.0 - omega_norm) * f_sample, -1  # above Nyquist: fold back, negate
    return omega_norm * f_sample, 1


f_alias, alias_sign = alias_frequency(F_SIGNAL, F_SAMPLE_BAD)

t_cont = np.linspace(0, WINDOW_S, N_CONT)
y_cont = np.sin(2 * np.pi * F_SIGNAL * t_cont)

t_ok = np.arange(0, WINDOW_S, 1 / F_SAMPLE_OK)
y_ok = np.sin(2 * np.pi * F_SIGNAL * t_ok)

t_bad = np.arange(0, WINDOW_S, 1 / F_SAMPLE_BAD)
y_bad = np.sin(2 * np.pi * F_SIGNAL * t_bad)

y_alias_cont = alias_sign * np.sin(2 * np.pi * f_alias * t_cont)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Left panel — adequate sampling: sample dots clearly track the 1,000 Hz wave
ax1.plot(t_cont * 1000, y_cont, color='steelblue', lw=1.2, alpha=0.5,
         label=f'True signal ({F_SIGNAL:.0f} Hz)')
ax1.stem(t_ok * 1000, y_ok, linefmt='tomato', markerfmt='ro', basefmt=' ',
         label=f'Samples at {F_SAMPLE_OK:.0f} Hz')
ax1.set_title(f'Adequate sampling: {F_SAMPLE_OK:.0f} Hz\n'
              f'(Nyquist = {F_SAMPLE_OK/2:.0f} Hz  >  {F_SIGNAL:.0f} Hz signal)')
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Amplitude')
ax1.legend(fontsize=9)
ax1.grid(True, linestyle=':', alpha=0.5)

# Right panel — aliasing: same samples, but they trace a false 200 Hz envelope
ax2.plot(t_cont * 1000, y_cont, color='steelblue', lw=1.2, alpha=0.25,
         label=f'True signal ({F_SIGNAL:.0f} Hz)')
ax2.plot(t_cont * 1000, y_alias_cont, color='darkorange', lw=2,
         label=f'Apparent alias ({f_alias:.0f} Hz)')
ax2.stem(t_bad * 1000, y_bad, linefmt='gray', markerfmt='go', basefmt=' ',
         label=f'Samples at {F_SAMPLE_BAD:.0f} Hz')
ax2.set_title(f'Aliasing: {F_SAMPLE_BAD:.0f} Hz sampling\n'
              f'(Nyquist = {F_SAMPLE_BAD/2:.0f} Hz  <  {F_SIGNAL:.0f} Hz signal)\n'
              f'{F_SIGNAL:.0f} Hz → alias at {f_alias:.0f} Hz')
ax2.set_xlabel('Time (ms)')
ax2.legend(fontsize=9)
ax2.grid(True, linestyle=':', alpha=0.5)

fig.suptitle(f'Nyquist–Shannon Sampling: {F_SIGNAL:.0f} Hz sinusoidal strain signal', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig10_nb1_aliasing_demo.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nNormalized frequency: {(F_SIGNAL / F_SAMPLE_BAD) % 1.0:.4f}"
      f"  →  alias at {f_alias:.0f} Hz  (sign: {alias_sign:+d})")

---
## 2 — Quantization Error and ADC Resolution

The ADC divides its input voltage range into 2^n discrete levels. Any voltage falling between two adjacent levels is rounded to the nearest — the rounding error is *quantization noise*, with RMS magnitude approximately LSB/√12. For a 16-bit ADC at 10 V full scale, LSB = 152.6 μV and RMS quantization noise is 44 μV. For a 24-bit ADC at the same range, LSB drops to 0.6 μV, at which point quantization noise is negligible compared to amplifier noise and gage thermal noise.

The table below computes LSB voltage, dynamic range in dB, and minimum detectable strain at GF = 2.0, V_exc = 5 V for four standard bit depths. Dynamic range scales at approximately 6 dB per bit (each bit doubles the count, adding 6.02 dB). Strain resolution halves with each additional bit — the relationship is log-linear.

In practice, ADC noise performance is specified by ENOB (effective number of bits), which accounts for harmonic distortion, thermal noise, and gain error. A 24-bit ADC may achieve only 20–22 ENOB. The minimum detectable strain at ENOB is what matters for system design, not the nominal bit count.

In [ ]:
# --- ADC resolution vs. bit depth ---
BITS_LIST = [8, 12, 16, 24]
V_RANGE   = 10.0  # ADC full-scale input range (V)
GF        = 2.0   # gage factor (dimensionless)
V_EXC     = 5.0   # bridge excitation voltage (V)


def strain_resolution_ue(lsb_volts, gage_factor=GF, v_exc=V_EXC):
    """Minimum resolvable strain (µε) for one LSB of a quarter-bridge output.

    A quarter bridge gives V_out = V_exc * GF * eps / 4, so one LSB step of
    ``lsb_volts`` corresponds to eps = 4 * lsb_volts / (V_exc * GF).
    """
    return 4 * lsb_volts / (v_exc * gage_factor) * 1e6


print(f"{'Bits':>6}  {'Levels':>8}  {'LSB (mV)':>10}  {'DR (dB)':>10}  {'Strain res @ GF=2, 5V':>22}")
print("-" * 62)
for bits in BITS_LIST:
    levels     = 2 ** bits
    lsb_volts  = V_RANGE / levels
    dr_dB      = 20 * np.log10(levels)
    eps_res_ue = strain_resolution_ue(lsb_volts)
    print(f"{bits:>6}  {levels:>8,}  {lsb_volts*1000:>10.4f}  {dr_dB:>10.1f}  {eps_res_ue:>18.2f} µε")

---
## Where to take this next

The two demonstrations above are deliberately minimal. Here are three concrete extensions that turn them into working tools for a real measurement program:

- **Confirm the alias in the frequency domain.** Take the FFT of the under-sampled series with `np.fft.rfft` and plot magnitude against `np.fft.rfftfreq(..., d=1/F_SAMPLE_BAD)`. The spectral peak lands at 200 Hz, not 1,000 Hz — the frequency-domain fingerprint of aliasing that the time-domain plot only hints at.
- **Model a real anti-aliasing filter.** Build a signal that contains both 200 Hz and 1,000 Hz content, apply a Butterworth low-pass (`scipy.signal.butter`) with its cutoff set below `F_SAMPLE_BAD / 2`, and show the 1,000 Hz component knocked down below the noise floor *before* sampling — the only place aliasing can ever be stopped.
- **Measure quantization noise against theory.** For each bit depth in the resolution table, quantize a full-scale sine to 2ⁿ levels, compute the residual RMS error, and compare it to the LSB/√12 prediction. Then subtract a fixed noise floor to see where nominal bits stop buying real resolution — the practical meaning of ENOB.